In [6]:
!pip install update

In [7]:
!pip install pyaudio faster-whisper numpy

   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 10.7 MB/s  0:00:00
   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
   ---- ----------------------------------- 2.4/19.2 MB 11.4 MB/s eta 0:00:02
   -------- ------------------------------- 4.2/19.2 MB 11.6 MB/s eta 0:00:02
   ---------- ----------------------------- 5.0/19.2 MB 8.2 MB/s eta 0:00:02
   -------------- ------------------------- 6.8/19.2 MB 8.2 MB/s eta 0:00:02
   ------------------ --------------------- 8.7/19.2 MB 8.3 MB/s eta 0:00:02
   ---------------------- ----------------- 10.7/19.2 MB 8.6 MB/s eta 0:00:01
   --------------------------- ------------ 13.1/19.2 MB 9.0 MB/s eta 0:00:01
   -------------------------------- ------- 15.5/19.2 MB 9.3 MB/s eta 0:00:01
   ---------------------------------- ----- 16.5/19.2 MB 8.9 MB/s eta 0:00:01
   ------------------------------------- -- 18.1/19.2 MB 8.8 MB/s eta 0:00:01
   -------

In [8]:
!pip install PyAudio==0.2.11

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Failed to build PyAudio


  error: subprocess-exited-with-error
  
  exit code: 1
  
  [8 lines of output]
  running bdist_wheel
  running build
  running build_py
  creating build\lib.win-amd64-cpython-313
  copying src\pyaudio.py -> build\lib.win-amd64-cpython-313
  running build_ext
  building '_portaudio' extension
  error: Microsoft Visual C++ 14.0 or greater is required. Get it with "Microsoft C++ Build Tools": https://visualstudio.microsoft.com/visual-cpp-build-tools/
  [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for PyAudio
error: failed-wheel-build-for-install

Failed to build installable wheels for some pyproject.toml based projects

PyAudio


In [ ]:
import pyaudio
import numpy as np
import queue
from faster_whisper import WhisperModel

# ==========================================
# 1. 오디오 및 큐(Queue) 설정
# ==========================================
RATE = 16000        # Whisper 모델이 요구하는 기본 샘플링 레이트 (16kHz)
CHUNK = 16000       # 1초 단위로 끊어서 처리 (16000 프레임 = 1초)
FORMAT = pyaudio.paInt16
CHANNELS = 1

# I/O 병목을 막기 위한 스레드 간 데이터 버퍼(Queue)
audio_queue = queue.Queue()

# ==========================================
# 2. PyAudio 콜백 함수 (생산자 역할)
# ==========================================
# 마이크에서 소리가 들어올 때마다 백그라운드 스레드에서 자동으로 실행됨
def audio_callback(in_data, frame_count, time_info, status):
    # 캡처된 바이트 데이터를 즉시 큐에 집어넣음 (I/O 블로킹 방지)
    audio_queue.put(in_data)
    return (None, pyaudio.paContinue)

# ==========================================
# 3. 모델 초기화
# ==========================================
print("모델을 로드하는 중입니다... (최초 실행 시 다운로드)")
# GPU(CUDA)를 사용할 수 있다면 'cuda', 아니면 'cpu' 사용
# compute_type="float16" (GPU) 또는 "int8" (CPU)로 최적화
# model = WhisperModel("small", device="cuda", compute_type="float16")
model = WhisperModel("small", device="cpu", compute_type="int8")
print("모델 로드 완료!")

# ==========================================
# 4. 오디오 스트림 시작
# ==========================================
p = pyaudio.PyAudio()
stream = p.open(format=FORMAT,
                channels=CHANNELS,
                rate=RATE,
                input=True,
                frames_per_buffer=CHUNK,
                stream_callback=audio_callback)

print("🎙️ 실시간 음성 인식을 시작합니다. (종료하려면 Ctrl+C)")
stream.start_stream()

# ==========================================
# 5. 메인 추론 루프 (소비자 역할)
# ==========================================
try:
    while True:
        # 큐에서 1초 분량의 오디오 조각(Chunk)을 꺼냄
        # 큐가 비어있으면 데이터가 들어올 때까지 대기(Block)함
        in_data = audio_queue.get()
        
        # Whisper 모델은 16kHz의 float32 형태의 NumPy 배열(-1.0 ~ 1.0)을 입력으로 받음
        audio_data = np.frombuffer(in_data, dtype=np.int16).astype(np.float32) / 32768.0
        
        # 모델 추론 진행
        # task="transcribe" (해당 언어로 자막 생성) 
        # task="translate" (무조건 영어로 번역하여 출력)
        segments, info = model.transcribe(
            audio_data, 
            beam_size=5, 
            task="translate", # 번역을 원할 경우. 자막만 원하면 "transcribe"
            language="ko"     # 입력 언어 (자동 감지하려면 None)
        )
        
        # 결과 즉시 출력
        for segment in segments:
            # 텍스트가 비어있지 않은 경우에만 화면에 Print
            text = segment.text.strip()
            if text:
                print(f"[{info.language}] {text}")
                
except KeyboardInterrupt:
    print("\n종료를 요청하셨습니다. 리소스를 정리합니다.")
finally:
    stream.stop_stream()
    stream.close()
    p.terminate()

C:\Users\USER\anaconda3\envs\opencv2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


모델을 로드하는 중입니다... (최초 실행 시 다운로드)


C:\Users\USER\anaconda3\envs\opencv2\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USER\.cache\huggingface\hub\models--Systran--faster-whisper-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


모델 로드 완료!
🎙️ 실시간 음성 인식을 시작합니다. (종료하려면 Ctrl+C)
[ko] Thank you for watching.
[ko] Thank you for watching.
[ko] Thank you for watching.
[ko] Thank you for watching.
[ko] Thank you for watching.
[ko] Thank you for watching.
